In [ ]:
# from serpapi import GoogleSearch
import time
import yaml

max_papers = 60
h_index = 16

ts = str(time.time())
with open("/home/config.yml", "r") as f:
    config = yaml.safe_load(f)

SERPAPI_KEY = config["SERPAPI_KEY"]
SCHOLAR-ID = config["SCHOLAR-ID"]

In [2]:
import serpapi
print(serpapi.__version__)
client = serpapi.Client(api_key=SERPAPI_KEY)

1.0.2


In [ ]:
import json
c = 0
results = []
while c < max_papers:
    res = client.search({
        "engine": "google_scholar_author",
        "SCHOLAR-ID": SCHOLAR-ID,
        "start": c,
        "hl": "en"
    })
    
    with open(f"data-serpapi/{ts}-papers-{c}-serpapi.json", "w", encoding="utf-8") as f:
        json.dump(res.as_dict(), f, indent=2, ensure_ascii=False)
    
    results.extend(res.get("articles", []))
    c += 20

with open(f"data-serpapi/{ts}-papers-all-serpapi.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

results

[{'title': 'A survey and experimental study on privacy-preserving trajectory data publishing',
  'link': 'https://scholar.google.com/citations?view_op=view_citation&hl=en&user=XKLsGcIAAAAJ&citation_for_view=XKLsGcIAAAAJ:0EnyYjriUFMC',
  'citation_id': 'XKLsGcIAAAAJ:0EnyYjriUFMC',
  'authors': 'F Jin, W Hua, M Francia, P Chao, ME Orlowska, X Zhou',
  'publication': 'IEEE Transactions on Knowledge and Data Engineering 35 (6), 5577-5596, 2022',
  'cited_by': {'value': 127,
   'link': 'https://scholar.google.com/scholar?oi=bibs&hl=en&cites=18203224539805011992',
   'serpapi_link': 'https://serpapi.com/search.json?cites=18203224539805011992&engine=google_scholar&hl=en',
   'cites_id': '18203224539805011992'},
  'year': '2022'},
 {'title': 'Crop management with the iot: An interdisciplinary survey',
  'link': 'https://scholar.google.com/citations?view_op=view_citation&hl=en&user=XKLsGcIAAAAJ&citation_for_view=XKLsGcIAAAAJ:hqOjcs7Dif8C',
  'citation_id': 'XKLsGcIAAAAJ:hqOjcs7Dif8C',
  'author

In [4]:
def low_citation_papers(papers, threshold=h_index):
    return [
        p for p in papers
        if p.get("cited_by", {}).get("value", 0) is not None and int(p.get("cited_by", {}).get("value", 0)) < threshold and int(p.get("cited_by", {}).get("value", 0)) > 0
    ]

low_papers = low_citation_papers(results)
low_papers

[{'title': 'HAMLET: A framework for Human-centered AutoML via Structured Argumentation',
  'link': 'https://scholar.google.com/citations?view_op=view_citation&hl=en&user=XKLsGcIAAAAJ&citation_for_view=XKLsGcIAAAAJ:Wp0gIr-vW9MC',
  'citation_id': 'XKLsGcIAAAAJ:Wp0gIr-vW9MC',
  'authors': 'M Francia, J Giovanelli, G Pisano',
  'publication': 'Future Generation Computer Systems 142, 182-194, 2023',
  'cited_by': {'value': 14,
   'link': 'https://scholar.google.com/scholar?oi=bibs&hl=en&cites=1263640081004138342',
   'serpapi_link': 'https://serpapi.com/search.json?cites=1263640081004138342&engine=google_scholar&hl=en',
   'cites_id': '1263640081004138342'},
  'year': '2023'},
 {'title': 'Social BI to understand the debate on vaccines on the Web and social media: unraveling the anti-, free, and pro-vax communities in Italy',
  'link': 'https://scholar.google.com/citations?view_op=view_citation&hl=en&user=XKLsGcIAAAAJ&citation_for_view=XKLsGcIAAAAJ:Y0pCki6q_DkC',
  'citation_id': 'XKLsGcIAA

In [ ]:
import json
import csv

def analyze_author():
    output = {}
    csv_rows = []
    for p in low_papers:
        title = p.get("title")
        citations = p.get("cited_by", {}).get("value", 0)
        cites_id = p.get("cited_by", {}).get("cites_id")

        print(f"\nProcessing: {title} ({citations} citations)")

        if not cites_id:
            continue

        output[title] = {
            "citations": citations,
            "cites_id": cites_id,
            "citing_papers": []
        }

        c = 0
        while c < citations:
            # Search citing papers
            p_results = client.search({
                "engine": "google_scholar",
                "cites": cites_id,
                "hl": "en",
                "start": c
            })
            c += 10

            organic_results = p_results.get("organic_results", [])
            output[title]["citing_papers"].extend(organic_results)


            # Prepare CSV rows
            for citing_paper in organic_results:
                citing_title = citing_paper.get("title", "")

                csv_rows.append({
                    "Paper title": title,
                    "Citing paper title": citing_title
                })


    # Save JSON
    with open(f"data-serpapi/{ts}-raw-citing-papers.json", "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)

    # Save CSV
    with open(f"data-serpapi/{ts}-scholar-citations-serpapi.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["Paper title", "Citing paper title"]
        )

        writer.writeheader()
        writer.writerows(csv_rows)

    return output


analyze_author()


Processing: HAMLET: A framework for Human-centered AutoML via Structured Argumentation (14 citations)

Processing: Social BI to understand the debate on vaccines on the Web and social media: unraveling the anti-, free, and pro-vax communities in Italy (14 citations)

Processing: Summarization and visualization of multi-level and multi-dimensional itemsets (13 citations)

Processing: Supporting the Generation of Data Narratives (13 citations)

Processing: Cost-based optimization of multistore query plans (11 citations)

Processing: DART: De-Anonymization of personal gazetteers through social trajectories (11 citations)

Processing: Colossal trajectory mining: a unifying approach to mine behavioral mobility patterns (10 citations)

Processing: The tell-tale cube (10 citations)

Processing: Augmented Business Intelligence. (9 citations)

Processing: Suggesting assess queries for interactive analysis of multidimensional data (8 citations)

Processing: Explaining cube measures through inte

{'HAMLET: A framework for Human-centered AutoML via Structured Argumentation': {'citations': 14,
  'cites_id': '1263640081004138342',
  'citing_papers': [{'position': 1,
    'title': 'Automated data processing and feature engineering for deep learning and big data applications: a survey',
    'result_id': 'MeGTkK1RkCwJ',
    'type': 'Html',
    'link': 'https://www.sciencedirect.com/science/article/pii/S2949715924000027',
    'snippet': 'Modern approach to artificial intelligence (AI) aims to design algorithms that learn directly from data. This approach has achieved impressive results and has contributed significantly …',
    'publication_info': {'summary': 'A Mumuni, F Mumuni - Journal of Information and Intelligence, 2025 - Elsevier',
     'authors': [{'name': 'A Mumuni'}, {'name': 'F Mumuni'}]},
    'resources': [{'title': 'sciencedirect.com',
      'file_format': 'HTML',
      'link': 'https://www.sciencedirect.com/science/article/pii/S2949715924000027'}],
    'inline_links': {'se

In [6]:
result = analyze_author("Matteo Francia, University of Bologna")

TypeError: analyze_author() takes 0 positional arguments but 1 was given